# Real-Time Multilingual Court Proceeding Summarization System
### Speech Recognition and Extractive Summarization Pipeline
**Course**: Speech and Language Processing — Capstone Project  
**Author / Student**: Capstone Researcher  

---

> **Core Design Principles**:
> 1. **Strictly Extractive**: Only selects real transcript sentences; never hallucinates or rewrites text.
> 2. **Legal Integrity**: Allegations and advocate submissions are never portrayed as binding court findings.
> 3. **Temporal Anchoring**: Every summary sentence links to its exact audio/video timestamp `[start_time, end_time]`.
> 4. **Assistive Tool Only**: Human-in-the-loop review is mandatory before formal legal reliance.

In [ ]:
# Cell 1: Environment Verification and Hardware Acceleration Check
import torch
import os
import random
import numpy as np

# Fix random seeds for full reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch Version: {torch.__version__}')
print(f'Device Detected: {device.upper()}')
if device == 'cuda':
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory Allocated: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('Running on CPU. CTranslate2 INT8 quantization will be utilized for high throughput.')

In [ ]:
# Cell 2: Install Pinned Dependencies and System FFmpeg
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1
!pip install -q \
    faster-whisper==1.2.1 \
    ctranslate2==4.8.3 \
    openai-whisper==20250625 \
    soundfile==0.14.0 \
    nltk==3.10.3 \
    scikit-learn==1.7.2 \
    networkx==3.4.2 \
    sentence-transformers==6.1.0 \
    jiwer==4.0.0 \
    rouge-score==0.1.2 \
    bert-score==0.3.13 \
    reportlab==5.0.0

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print('All environment dependencies installed successfully!')

In [ ]:
# Cell 3: Phase 1 & 2 - Video Audio Extraction to 16 kHz Mono WAV
import subprocess
from pathlib import Path

def extract_audio_ffmpeg(video_path: str, output_wav: str):
    Path(output_wav).parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        'ffmpeg', '-y', '-i', video_path,
        '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1',
        output_wav
    ]
    subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
    print(f'Extracted 16kHz WAV -> {output_wav}')

# Example invocation (adjust path to uploaded video)
test_video = 'Dataset/5) Movie CLIP - Automotive Expert (1992) HD.mp4'
test_wav = 'data/extracted_audio/sample_court.wav'
if Path(test_video).exists():
    extract_audio_ffmpeg(test_video, test_wav)

In [ ]:
# Cell 4: Phase 3 - Audio Preprocessing (Loudness Norm & Conservative Filtering)
def preprocess_audio(input_wav: str, output_wav: str):
    Path(output_wav).parent.mkdir(parents=True, exist_ok=True)
    # Standard EBU R128 Loudness Normalization (-23 LUFS) and bandpass filtering
    filter_expr = 'loudnorm=I=-23:TP=-1.5:LRA=11,highpass=f=80,lowpass=f=7500'
    cmd = ['ffmpeg', '-y', '-i', input_wav, '-af', filter_expr, '-ar', '16000', '-ac', '1', output_wav]
    subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
    print(f'Preprocessed audio normalized to -23 LUFS -> {output_wav}')

clean_wav = 'data/preprocessed_audio/sample_court_clean.wav'
if Path(test_wav).exists():
    preprocess_audio(test_wav, clean_wav)

In [ ]:
# Cell 5: Phase 4 & 5 - Multilingual ASR with Language Detection
from faster_whisper import WhisperModel

compute_type = 'float16' if device == 'cuda' else 'int8'
model_size = 'tiny'  # Options: 'tiny', 'base', 'small'
print(f'Loading faster-whisper [{model_size}] on {device} ({compute_type})...')
asr_model = WhisperModel(model_size, device=device, compute_type=compute_type)

def transcribe_court_audio(wav_path: str):
    segments, info = asr_model.transcribe(
        wav_path,
        beam_size=5,
        vad_filter=True,
        condition_on_previous_text=False
    )
    print(f'Detected Language: {info.language.upper()} (Confidence: {info.language_probability*100:.1f}%)')
    print(f'Audio Duration: {info.duration:.1f} seconds')
    
    seg_list = []
    for s in segments:
        seg_list.append({
            'id': s.id,
            'start': round(s.start, 2),
            'end': round(s.end, 2),
            'text': s.text.strip()
        })
    return seg_list, info.language

if Path(clean_wav).exists():
    raw_segments, detected_lang = transcribe_court_audio(clean_wav)
    print(f'Extracted {len(raw_segments)} raw ASR speech segments.')

In [ ]:
# Cell 6: Phase 6 - Legal Cleaning and Boundary-Aware Sentence Segmentation
import re

def segment_and_clean_sentences(raw_segments):
    sentences = []
    sent_id = 1
    for seg in raw_segments:
        text = seg['text']
        # Clean speech filler disfluencies without altering semantics
        text = re.sub(r'\b(um+|uh+|ah+)\b', '', text, flags=re.IGNORECASE).strip()
        text = re.sub(r'\s+', ' ', text)
        if len(text.split()) < 3:
            continue
        sentences.append({
            'sentence_id': sent_id,
            'start_time': seg['start'],
            'end_time': seg['end'],
            'text': text,
            'word_count': len(text.split())
        })
        sent_id += 1
    return sentences

if 'raw_segments' in locals():
    sentences = segment_and_clean_sentences(raw_segments)
    print(f'Constructed {len(sentences)} clean timestamped candidate sentences.')

In [ ]:
# Cell 7: Phase 7 & 8 - Extractive Summarization Engine with Legal Domain Boosting
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

LEGAL_KEYWORDS = ['held', 'ruled', 'order', 'bail', 'evidence', 'witness', 'testimony', 'section', 'ipc', 'verdict', 'acquitted', 'convicted']
ALLEGATION_MARKERS = ['claims', 'alleges', 'argues', 'contends', 'alleged']

def textrank_summarize(sentences, ratio=0.25, legal_boost=True):
    texts = [s['text'] for s in sentences]
    vec = TfidfVectorizer(stop_words='english')
    tfidf = vec.fit_transform(texts)
    sim = cosine_similarity(tfidf, tfidf)
    
    # Build Graph
    G = nx.Graph()
    for i in range(len(sentences)):
        G.add_node(i)
    for i in range(len(sentences)):
        for j in range(i+1, len(sentences)):
            if sim[i, j] > 0.05:
                G.add_edge(i, j, weight=sim[i, j])
                
    scores = nx.pagerank(G, alpha=0.85, weight='weight')
    
    scored_list = []
    for i, s in enumerate(sentences):
        sc = scores.get(i, 0.0)
        t_low = s['text'].lower()
        is_ruling = any(kw in t_low for kw in LEGAL_KEYWORDS)
        is_allegation = any(al in t_low for al in ALLEGATION_MARKERS)
        
        if legal_boost and is_ruling:
            sc *= 2.0
        elif legal_boost and is_allegation:
            sc *= 0.85
            
        category = 'JUDICIAL_RULING' if is_ruling else ('ADVOCATE_ALLEGATION' if is_allegation else 'TESTIMONY')
        scored_list.append({'orig_idx': i, 'sent': s, 'score': sc, 'category': category})
        
    # Select top k by score
    k = max(2, int(len(sentences) * ratio))
    top_k = sorted(scored_list, key=lambda x: x['score'], reverse=True)[:k]
    # Enforce strict chronological order
    top_k_chronological = sorted(top_k, key=lambda x: x['orig_idx'])
    return top_k_chronological

if 'sentences' in locals():
    summary = textrank_summarize(sentences, ratio=0.25, legal_boost=True)
    print(f'\n=== EXTRACTIVE COURT SUMMARY ({len(summary)} Key Moments) ===\n')
    for idx, item in enumerate(summary, 1):
        s = item['sent']
        print(f"{idx}. [{s['start_time']:05.1f}s - {s['end_time']:05.1f}s] [{item['category']}]")
        print(f"   {s['text']}\n")

In [ ]:
# Cell 8: Phase 9 - Quantitative Evaluation with ROUGE and Sentence F1
from rouge_score import rouge_scorer

def evaluate_summary_rouge(hypothesis_text, reference_text):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference_text, hypothesis_text)
    return {
        'ROUGE-1': round(scores['rouge1'].fmeasure, 4),
        'ROUGE-2': round(scores['rouge2'].fmeasure, 4),
        'ROUGE-L': round(scores['rougeL'].fmeasure, 4)
    }

hyp_text = ' '.join(x['sent']['text'] for x in summary) if 'summary' in locals() else ''
ref_text = 'Miss Vito automotive mechanics expert witness tire marks exculpatory evidence Buick Skylark dismissal.'
metrics = evaluate_summary_rouge(hyp_text, ref_text)
print('Automated Evaluation Metrics on Ground Truth:')
for k, v in metrics.items():
    print(f' {k}: {v*100:.2f}%')